### NIPS 2014~ 2019

In [1]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd

def create_database(DB_PATH):
    """Create the database and the Conference table."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Conference (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        Title TEXT NOT NULL,
        Author TEXT NOT NULL,
        PDF_Link TEXT,
        Code_URL TEXT,
        Conference_Name TEXT NOT NULL
    )
    ''')

    print("Conference 테이블이 생성되었습니다.")
    conn.commit()
    conn.close()

def save_to_database(df, conference_name, DB_PATH):
    conn = sqlite3.connect(DB_PATH, timeout=10)
    cursor = conn.cursor()

    try:
        for _, row in df.iterrows():  # ✅ iterrows() 사용하여 DataFrame의 각 행을 처리
            # 중복 데이터 확인
            cursor.execute('''
            SELECT 1 FROM Conference WHERE Title = ? AND Author = ? AND Conference_Name = ?
            ''', (row['title'], row['authors'], conference_name))
            result = cursor.fetchone()

            if not result:
                cursor.execute('''
                INSERT INTO Conference (Title, Author, PDF_Link, Code_URL, Conference_Name)
                VALUES (?, ?, ?, ?, ?)
                ''', (row['title'], row['authors'], row['pdf_link'], row['code_url'], conference_name))

        conn.commit()
        print(f"{len(df)}개의 논문이 {conference_name}에 저장되었습니다.")
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        conn.close()

In [3]:
import re
def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 ICDM 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 리스트가 포함된 섹션 찾기
    for entry in soup.find_all("li", class_="entry inproceedings"):
        # 제목 찾기
        title_tag = entry.find("span", class_="title")
        title_text = title_tag.text.strip() if title_tag else "Unknown"

        # 저자 찾기
        authors_tags = entry.find_all("span", itemprop="author")
        authors_list = [author.find("span", itemprop="name").text.strip() for author in authors_tags]
        authors_cleaned = ", ".join(authors_list)

        # DOI 링크 찾기 (electronic edition via DOI)
        doi_tag = entry.find("a", href=re.compile(r"doi\.org"))
        if doi_tag and doi_tag.get("href"):
            pdf_link = doi_tag["href"]
        else:
            pdf_link = None

        # 논문 정보 추가
        papers.append({
            "title": title_text,
            "authors": authors_cleaned,
            "pdf_link": pdf_link,
            'code_url': None,
            "conference_name": conference_name
        })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers

In [4]:
url = 'https://dblp.org/db/conf/nips/nips2019.html'
DB_PATH = "con_db/NIPS_conference_2019.db"
conference_name = 'NIPS 2019'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [5]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/NIPS_2019_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [6]:
df_papers = get_www_papers('html/NIPS_2019_accepted_papers.html', conference_name)

In [7]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Multimodal Model-Agnostic Meta-Learning via Ta...,"Risto Vuorio, Shao-Hua Sun, Hexiang Hu, Joseph...",None,None,NIPS 2019
1,ViLBERT: Pretraining Task-Agnostic Visiolingui...,"Jiasen Lu, Dhruv Batra, Devi Parikh, Stefan Lee",None,None,NIPS 2019
2,Stochastic Shared Embeddings: Data-driven Regu...,"Liwei Wu, Shuqing Li, Cho-Jui Hsieh, James L. ...",None,None,NIPS 2019
3,Unsupervised Scale-consistent Depth and Ego-mo...,"Jiawang Bian, Zhichao Li, Naiyan Wang, Huangyi...",None,None,NIPS 2019
4,Zero-shot Learning via Simultaneous Generating...,"Hyeonwoo Yu, Beomhee Lee",None,None,NIPS 2019


In [8]:
save_to_database(df_papers, conference_name, DB_PATH)

1427개의 논문이 NIPS 2019에 저장되었습니다.


# 2018

In [9]:
url = 'https://dblp.org/db/conf/nips/nips2018.html'
DB_PATH = "con_db/NIPS_conference_2018.db"
conference_name = 'NIPS 2018'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [10]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/NIPS_2018_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [11]:
df_papers = get_www_papers('html/NIPS_2018_accepted_papers.html', conference_name)

In [12]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Efficient Algorithms for Non-convex Isotonic R...,Francis R. Bach,None,None,NIPS 2018
1,Structure-Aware Convolutional Neural Networks.,"Jianlong Chang, Jie Gu, Lingfeng Wang, Gaofeng...",None,None,NIPS 2018
2,Kalman Normalization: Normalizing Internal Rep...,"Guangrun Wang, Jiefeng Peng, Ping Luo, Xinjian...",None,None,NIPS 2018
3,HOGWILD!-Gibbs can be PanAccurate.,"Constantinos Daskalakis, Nishanth Dikkala, Sid...",None,None,NIPS 2018
4,Text-Adaptive Generative Adversarial Networks:...,"Seonghyeon Nam, Yunji Kim, Seon Joo Kim",None,None,NIPS 2018


In [13]:
save_to_database(df_papers, conference_name, DB_PATH)

1010개의 논문이 NIPS 2018에 저장되었습니다.


# 2017

In [14]:
url = 'https://dblp.org/db/conf/nips/nips2017.html'
DB_PATH = "con_db/NIPS_conference_2017.db"
conference_name = 'NIPS 2017'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [15]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/NIPS_2017_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [16]:
df_papers = get_www_papers('html/NIPS_2017_accepted_papers.html', conference_name)

In [17]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,"Wider and Deeper, Cheaper and Faster: Tensoriz...","Zhen He, Shaobing Gao, Liang Xiao, Daxue Liu, ...",None,None,NIPS 2017
1,Concentration of Multilinear Functions of the ...,"Constantinos Daskalakis, Nishanth Dikkala, Gau...",None,None,NIPS 2017
2,Deep Subspace Clustering Networks.,"Pan Ji, Tong Zhang, Hongdong Li, Mathieu Salzm...",None,None,NIPS 2017
3,Attentional Pooling for Action Recognition.,"Rohit Girdhar, Deva Ramanan",None,None,NIPS 2017
4,On the Consistency of Quick Shift.,Heinrich Jiang,None,None,NIPS 2017


In [18]:
save_to_database(df_papers, conference_name, DB_PATH)

679개의 논문이 NIPS 2017에 저장되었습니다.


# 2016

In [19]:
url = 'https://dblp.org/db/conf/nips/nips2016.html'
DB_PATH = "con_db/NIPS_conference_2016.db"
conference_name = 'NIPS 2016'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [20]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/NIPS_2016_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [21]:
df_papers = get_www_papers('html/NIPS_2016_accepted_papers.html', conference_name)

In [22]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Scan Order in Gibbs Sampling: Models in Which ...,"Bryan D. He, Christopher De Sa, Ioannis Mitlia...",None,None,NIPS 2016
1,Deep ADMM-Net for Compressive Sensing MRI.,"Yan Yang, Jian Sun, Huibin Li, Zongben Xu",None,None,NIPS 2016
2,A scaled Bregman theorem with applications.,"Richard Nock, Aditya Krishna Menon, Cheng Soon...",None,None,NIPS 2016
3,Swapout: Learning an ensemble of deep architec...,"Saurabh Singh, Derek Hoiem, David A. Forsyth",None,None,NIPS 2016
4,On Regularizing Rademacher Observation Losses.,Richard Nock,None,None,NIPS 2016


In [23]:
save_to_database(df_papers, conference_name, DB_PATH)

569개의 논문이 NIPS 2016에 저장되었습니다.


# 2015

In [24]:
url = 'https://dblp.org/db/conf/nips/nips2015.html'
DB_PATH = "con_db/NIPS_conference_2015.db"
conference_name = 'NIPS 2015'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [25]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/NIPS_2015_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [26]:
df_papers = get_www_papers('html/NIPS_2015_accepted_papers.html', conference_name)

In [27]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Double or Nothing: Multiplicative Incentive Me...,"Nihar Bhadresh Shah, Denny Zhou",None,None,NIPS 2015
1,Learning with Symmetric Label Noise: The Impor...,"Brendan van Rooyen, Aditya Krishna Menon, Robe...",None,None,NIPS 2015
2,Algorithmic Stability and Uniform Generalization.,Ibrahim M. Alabdulmohsin,None,None,NIPS 2015
3,Adaptive Low-Complexity Sequential Inference f...,"Theodoros Tsiligkaridis, Keith W. Forsythe",None,None,NIPS 2015
4,Covariance-Controlled Adaptive Langevin Thermo...,"Xiaocheng Shang, Zhanxing Zhu, Benedict J. Lei...",None,None,NIPS 2015


In [28]:
save_to_database(df_papers, conference_name, DB_PATH)

403개의 논문이 NIPS 2015에 저장되었습니다.


# 2014

In [34]:
url = 'https://dblp.org/db/conf/nips/mlini2014.html'
DB_PATH = "con_db/NIPS_conference_2014.db"
conference_name = 'NIPS 2014'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [35]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/NIPS_2014_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [36]:
df_papers = get_www_papers('html/NIPS_2014_accepted_papers.html', conference_name)

In [37]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Multi-Task Learning for Interpretation of Brai...,"Seyed Mostafa Kia, Sandro Vega-Pons, Emanuele ...",https://doi.org/10.1007/978-3-319-45174-9_1,None,NIPS 2014
1,The New Graph Kernels on Connectivity Networks...,"Biao Jie, Xi Jiang, Chen Zu, Daoqiang Zhang",https://doi.org/10.1007/978-3-319-45174-9_2,None,NIPS 2014
2,Mapping Tractography Across Subjects.,"Thien Bao Nguyen, Emanuele Olivetti, Paolo Ave...",https://doi.org/10.1007/978-3-319-45174-9_3,None,NIPS 2014
3,Automated Speech Analysis for Psychosis Evalua...,"Facundo Carrillo, Natalia Mota, Mauro Copelli,...",https://doi.org/10.1007/978-3-319-45174-9_4,None,NIPS 2014
4,Combining Different Modalities in Classifying ...,"Shunan Zhao, Frank Rudzicz",https://doi.org/10.1007/978-3-319-45174-9_5,None,NIPS 2014


In [38]:
save_to_database(df_papers, conference_name, DB_PATH)

13개의 논문이 NIPS 2014에 저장되었습니다.
